# Третий этап: BGE Small, dense и hybrid retrieval

Ноутбук показывает анализ длины SciFact, локальные embeddings `BAAI/bge-small-en-v1.5`, поиск в Qdrant, RRF и общие метрики. Внешний API и платный ключ не используются.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Находим корень проекта независимо от того, откуда открыт ноутбук.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден pyproject.toml')
os.chdir(PROJECT_ROOT)

def run_cli(*arguments: str, env: dict[str, str] | None = None) -> str:
    """Запустить тот же CLI, который доступен из терминала."""
    result = subprocess.run(
        [sys.executable, '-m', 'rag_playground.cli', *arguments],
        cwd=PROJECT_ROOT, env=env, check=True, capture_output=True, text=True,
    )
    return result.stdout.strip()

print(f'Корень: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')

Корень: /Users/e.ilyina/Desktop/Портфолио/RAG
Python: 3.13.0


## 1. Конфигурация

Проверяем модель, лимит, Qdrant-коллекцию и RRF.

In [2]:
from rag_playground.config import Settings

settings = Settings.from_env(PROJECT_ROOT / '.env')
safe_settings = {
    'model': settings.dense_model_name,
    'device': settings.dense_device,
    'cache_dir': str(settings.dense_cache_dir),
    'max_length': settings.dense_max_length,
    'collection': settings.dense_collection,
    'qdrant_url': settings.qdrant_url,
    'rrf_k': settings.rrf_k,
}
print(json.dumps(safe_settings, ensure_ascii=False, indent=2))

{
  "model": "BAAI/bge-small-en-v1.5",
  "device": "cpu",
  "cache_dir": "data/models/fastembed",
  "max_length": 512,
  "collection": "scifact-documents-bge-small-en-v1-5",
  "qdrant_url": "http://localhost:6333",
  "rrf_k": 60
}


## 2. Анализ длины корпуса

Длина считается реальным tokenizer'ом BGE до обрезания.

In [3]:
# Модель уже находится в локальном кэше, поэтому API-запросов нет.
analysis_output = run_cli('dense-analyze')
print(analysis_output)

{
  "model_name": "BAAI/bge-small-en-v1.5",
  "max_length": 512,
  "documents": 5183,
  "min_tokens": 70,
  "mean_tokens": 337.194,
  "median_tokens": 316.0,
  "p95_tokens": 567.9,
  "max_tokens": 1939,
  "truncated_documents": 455,
  "truncated_percent": 8.779,
  "longest_documents": [
    {
      "doc_id": "26067999",
      "title": "Screening for Lung Cancer: U.S. Preventive Services Task Force Recommendation Statement",
      "tokens": 1939
    },
    {
      "doc_id": "10749308",
      "title": "Placebo-Controlled Trials and Active-Control Trials in the Evaluation of New Treatments. Part 1: Ethical and Scientific Issues",
      "tokens": 1919
    },
    {
      "doc_id": "15521377",
      "title": "Keeping your senescent cells under control",
      "tokens": 1526
    },
    {
      "doc_id": "5114940",
      "title": "Pharmacological interventions for smoking cessation: an overview and network meta-analysis.",
      "tokens": 1474
    },
    {
      "doc_id": "27768226",
      "ti

## 3. Возобновляемая индексация

Коллекция уже заполнена. Повторный запуск должен пропустить все 5 183 `doc_id` и не считать embeddings повторно.

In [4]:
print(run_cli('dense-index'))

Collection 'scifact-documents-bge-small-en-v1-5' already exists; already indexed documents will be skipped.
{
  "collection_name": "scifact-documents-bge-small-en-v1-5",
  "documents_read": 5183,
  "documents_skipped": 5183,
  "documents_embedded": 0,
  "documents_upserted": 0,
  "truncated_documents": 0,
  "duration_seconds": 0.178591,
  "final_point_count": 5183
}


## 4. Коллекция и один поиск

Векторы имеют 384 измерения, а Qdrant сравнивает их по cosine similarity.

In [5]:
print('Коллекция:')
print(run_cli('dense-info'))

# Длинный abstract сокращаем только для экрана; в Qdrant он хранится целиком.
result = json.loads(run_cli(
    'dense-search', 'Vitamin D supplementation improves bone mineral density.',
    '--top-k', '3',
))
print(f"\nEmbedding: {result['embedding_latency_ms']} мс; Qdrant: {result['search_latency_ms']} мс")
for hit in result['hits']:
    print(f"#{hit['rank']} score={hit['score']:.4f} doc_id={hit['doc_id']} | {hit['title']}")

Коллекция:


{
  "collection_name": "scifact-documents-bge-small-en-v1-5",
  "collection_exists": true,
  "point_count": 5183,
  "vector_size": 384,
  "distance": "Cosine",
  "status": "green",
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "max_length": 512
}



Embedding: 9.441 мс; Qdrant: 8.103 мс
#1 score=0.7943 doc_id=16256507 | Relationship between serum parathyroid hormone levels, vitamin D sufficiency, and calcium intake.
#2 score=0.7902 doc_id=4462079 | Estimation of optimal serum concentrations of 25-hydroxyvitamin D for multiple health outcomes.
#3 score=0.7875 doc_id=38551172 | Mammographic density, plasma vitamin D levels and risk of breast cancer in postmenopausal women.


## 5. Batch dense, RRF и метрики

Повторяем dense-run по 300 test-запросам, после чего объединяем его с BM25.

In [6]:
print(run_cli('dense-run', '--split', 'test', '--top-k', '100'))
hybrid_output = json.loads(run_cli('hybrid-run', '--split', 'test', '--top-k', '100'))
print(json.dumps({key: value for key, value in hybrid_output.items() if key != 'results'}, ensure_ascii=False, indent=2))

{
  "ranking": "reports/results/runs/dense_bge_small_test_top100.jsonl",
  "queries": "reports/results/runs/dense_bge_small_test_top100_queries.jsonl",
  "manifest": "reports/results/runs/dense_bge_small_test_top100_manifest.json"
}


{
  "hybrid_ranking": "reports/results/runs/hybrid_rrf_test_top100.jsonl",
  "hybrid_manifest": "reports/results/runs/hybrid_rrf_test_top100_manifest.json",
  "metrics": "reports/results/metrics/document_retrieval_test.json"
}


In [7]:
# Выводим компактную таблицу ключевых метрик.
metrics = hybrid_output['results']
print(f"{'method':<18} {'Recall@10':>10} {'MRR@10':>10} {'nDCG@10':>10} {'Recall@100':>12}")
for method, values in metrics.items():
    print(
        f"{method:<18} {values['recall@10']:>10.4f} {values['mrr@10']:>10.4f} "
        f"{values['ndcg@10']:>10.4f} {values['recall@100']:>12.4f}"
    )

method              Recall@10     MRR@10    nDCG@10   Recall@100
bm25                   0.8173     0.6464     0.6826       0.9282
dense_bge_small        0.8362     0.6822     0.7127       0.9417
hybrid_rrf             0.8612     0.6910     0.7268       0.9683


## 6. Проверки

Обычный Pytest не запускает внешние сервисы; Qdrant integration включается явно.

In [8]:
def run_tool(*arguments: str, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(
        list(arguments), cwd=PROJECT_ROOT, env=env, check=True,
        capture_output=True, text=True,
    )
    return (result.stdout + result.stderr).strip()

print(run_tool(sys.executable, '-m', 'ruff', 'check', 'src', 'tests'))
print(run_tool(sys.executable, '-m', 'pytest'))
integration_env = os.environ.copy()
integration_env['RUN_QDRANT_INTEGRATION'] = '1'
print(run_tool(
    sys.executable, '-m', 'pytest', 'tests/integration/test_qdrant_dense.py',
    env=integration_env,
))

All checks passed!


ss...........................                                            [100%]
27 passed, 2 skipped in 1.80s


.                                                                        [100%]
1 passed in 2.52s


## 7. Итог

BGE Small улучшает документный baseline относительно BM25. Hybrid RRF даёт лучшие Recall@10/100 и nDCG@10/100. На следующем этапе chunking уберёт потерю конца у 455 длинных документов.